In [ ]:
from pyspark.sql import functions as F
from delta.tables import DeltaTable
from pyspark.sql.window import Window

In [ ]:
# ---> LOAD PROJECT UTILITIES & INIT NOTEBOOK WIDGETS <---

%run /Workspace/consolidated_pipeline/1_setup/utilities


In [ ]:
dbutils.widget.text('catalog', 'fmcg', 'Catalog')
dbutils.widgets.text('data_source', 'gross_price', 'Data Source')

catalog = dbutils.widgets.get('catalog')
data_source = dbutils.widgets.get('data_source')

base_path = f's3://sports-consolidated-dp/{data_source}/*.csv'

In [ ]:
# BRONZE

df = (
    spark.read.format('csv')
        .option('header', True)
        .option('inferSchema', True)
        .load(base_path)
        .withColumn('read_timestamp', F.current_timestamp())
        .select('*', _'metadata.file_name', '_metadata.file_size')
)

In [ ]:
df.write\
    .format('delta') \
    .option('delta.enableDataChange', 'true')\
    .mode('overwrite') \
    .saveAsTable(f'{catalog}.{bronze_schema}.{data_source}')

In [ ]:
# SILVER
df_bronze = spark.sql(f'SELECT * from {catalog}.{bronze_schema}.{data_source};')

In [ ]:
# ---> NORMALIZE MONTH FIELD <---

df_bronze.select('month').distinct().show()

# +----------+
# |     month|
# +----------+
# |2025/07/01|
# |01/08/2025|
# |2025/09/01|
# |2025-10-01|
# |2025-11-01|
# |2025-12-01|
# |2025-07-01|
# |2025-08-01|
# |2025-09-01|
# |2025/11/01|
# |2025/08/01|
# |01-09-2025|
# |2025/10/01|
# |01/12/2025|
# |01/09/2025|
# |01-12-2025|
# |01-08-2025|
# |01/10/2025|
# +----------+

In [ ]:
# --> Parse 'month' from multiple possible formats

df_silver = df_bronze.withColumn(
    'month',
    F.coalesce(
        F.try_to_date(F.col('month'), 'yyyy/MM/dd'),
        F.try_to_date(F.col('month'), 'dd/MM/yyyy'),
        F.try_to_date(F.col('month'), 'yyyy-MM-dd'),
        F.try_to_date(F.col('month'), 'dd-MM-yyyy')
    )
)

In [ ]:
df.silver.select('month').unique().show()

# +----------+
# |     month|
# +----------+
# |2025-08-01|
# |2025-10-01|
# |2025-12-01|
# |2025-09-01|
# |2025-11-01|
# |2025-07-01|
# +----------+

In [ ]:
# Validate the gross_price column, convert only valid numeric values to double,
# Convert negative prices to positive, and replace al lnon-numeric values with 0.

# '^-?\d+(\.\d+)?$' - Regex to validate whether a string is a positive or negative integer, or decimal number
df_silver = df_silver.withColumn(
    'gross_price',
    F.when(F.col('gross_price').rlike(r'^-?\d+(\.\d+)?$'),
    F.when(F.col('gross_price').cast('double') < 0, -1 * F.col('gross_price').cast('double'))
    .otherwise(F.col('gross_price').cast('double'))
    ).otherwise(0)
)

In [ ]:
# Enrich the silver dataset by performing inner join with the product table to fetch the correct product_code for each product_id

df_products = spark.table('fmcg.silver.products')
df_joined = df_silver.join(df_products.select('product_id', 'product_code'), on='product_id', how='inner')
df_joined = df_joined.select('product_id', 'product_code', 'month', 'gross_price', 'read_timestamp', 'file_name', 'file_size')


In [ ]:
df_joined.write\
    .format('delta') \
    .option('delta.enableChangeDataFeed', 'true') \
    .option('mergeSchema', 'true') \
    .mode('overwrite') \
    .saveAsTable(f'{catalog}.{silver_schema}.{data_source}')

In [ ]:
# ---> GOLD <---
df_silver = spark.sql(f'SELECT * FROM {catalog}.{silver_schema}.{data_source};')

In [ ]:
df_gold = df_silver.select('product_code', 'momnth', 'gross_price')
df_gold.show(5)
# +--------------------+----------+-----------+
# |        product_code|     month|gross_price|
# +--------------------+----------+-----------+
# |e91ba9d665f90254d...|2025-07-01|       84.0|
# |e91ba9d665f90254d...|2025-08-01|        0.0|
# |e91ba9d665f90254d...|2025-09-01|       84.0|
# |e91ba9d665f90254d...|2025-10-01|       83.0|
# |e91ba9d665f90254d...|2025-11-01|       83.0|
# +--------------------+----------+-----------+
# only showing top 5 rows

In [ ]:
df_gold.write('delta') \
    .format('delta') \
    .option('delta.enableChangeDataFeed', 'true') \
    .mode('overwrite') \
    .saveAsTable(f'{catalog}.{gold_schmea}.sb_dim_{data_source}')

In [ ]:
# ---> MERGING DATA SOURCE WITH PARENT <---
df_gold_price = spark.table('fmcg.gold.sb_dim_gross_price')
df_gold_price.show(5)
# +--------------------+----------+-----------+
# |        product_code|     month|gross_price|
# +--------------------+----------+-----------+
# |e91ba9d665f90254d...|2025-07-01|       84.0|
# |e91ba9d665f90254d...|2025-08-01|        0.0|
# |e91ba9d665f90254d...|2025-09-01|       84.0|
# |e91ba9d665f90254d...|2025-10-01|       83.0|
# |e91ba9d665f90254d...|2025-11-01|       83.0|
# +--------------------+----------+-----------+
# only showing top 5 rows

In [ ]:
# --> Get the price for eah product_code (aggregated by year) <---
df_gold_price = (
    df_gold_price
    .withColumn('year', F.year('month'))
    .withColumn('is_zero', F.when(F.col('gross_price') == 0, 1).otherwise(0))
)

w = (
    Window
    .partitionBy('product_code', 'year')
    .orderBy(F.col('is_zero'), F.col('month').desc())
)

df_gold_latest_price = (
    df_gold_price
        .withColumn('rnk', F.row_number().over(w))
        .filter(F.col('rnk') == 1)
)

In [ ]:
# Take required cols

df_gold_latest_price = df_gold_latest_price.select('product_code', 'year', 'gross_price').withColumnRenamed('gross_price', 'price_inr').select('product_code', 'price_inr', 'year')

df_gold_latest_price = df_gold_latest_price.withColumn('year', F.col('year').cast('string'))

df_gold_latest_price.show(5)

# +--------------------+---------+----+
# |        product_code|price_inr|year|
# +--------------------+---------+----+
# |062f5574bbdf4386b...|    281.0|2025|
# |0cb7b2f42657b625f...|    100.0|2025|
# |102628255d24304d6...|     86.0|2025|
# |2e387cef1424d6e7b...|    108.0|2025|
# |3cab59f0592428527...|    493.0|2025|
# +--------------------+---------+----+
# only showing top 5 rows

In [ ]:
delta_table = DeltaTable.forName(spark, 'fmcg.gold.dim_gross_price')

# UPSERT DELTA_TABLE

delta_table.alias('parent').merge(
    source=df_gold_latest_price.alias('child'),
    condition='parent.product_code = child.product_code'
).whenMatchedUpdate(
    set={
        'price_inr': 'child.price_inr',
        'year': 'child.year'
    }
).whenNotMatchedInsert(
    values={
        'product_code': 'child.product_code',
        'price_inr': 'child.price_inr',
        'year': 'child.year'
    }
).execute()